# 🧠 CHEATSHEET MASIVO — Redes Neuronales (Keras + PyTorch)
## IES de Teis · UD4 Sistemas de Aprendizaxe Automáticos
---
> **CÓMO USAR ESTE NOTEBOOK:**  
> Cada sección está lista para copiar-pegar o ejecutar directamente.  
> Todos los bloques incluyen comentarios explicativos orientados a examen.

### 📋 ÍNDICE RÁPIDO
| # | Sección | Contenido |
|---|---------|-----------|
| 1 | Imports | Todo lo que necesitas importar |
| 2 | EDA & Preprocesado | Pandas, nulls, outliers, codificación, escalado |
| 3 | Pipeline Keras — Datos Tabulares | Clasificación binaria completa (estilo Proyecto Crédito) |
| 4 | Pipeline Keras — Imágenes | CNN completa (estilo Fashion-MNIST) |
| 5 | EarlyStopping & class_weight | Técnicas avanzadas de entrenamiento |
| 6 | Métricas & Evaluación | Accuracy, F1, ROC-AUC, Balanced Accuracy |
| 7 | Gráficas | Loss curves, confusion matrix, comparativa de modelos |
| 8 | Modelos Sklearn | LR, RF, SVM, KNN + Cross-Validation |
| 9 | PyTorch | Module, Sequential, DataLoader, bucle de entrenamiento |
| 10 | Cálculo de Parámetros | Fórmulas para Dense y Conv2D |
| 11 | Debugging | Errores comunes y cómo arreglarlos |
| 12 | Plantillas Rápidas | Templates copiables para el examen |


---
# 📦 SECCIÓN 1 — IMPORTS COMPLETOS

In [ ]:
# ══════════════════════════════════════════════
#  IMPORTS UNIVERSALES — pega esto al inicio de cualquier notebook
# ══════════════════════════════════════════════
import os, warnings, math
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'   # silencia logs de TF
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Semilla global para reproducibilidad
seed = 42
np.random.seed(seed)

print("✅ Imports básicos OK")


In [ ]:
# ══════════════════════════════════════════════
#  IMPORTS KERAS (TensorFlow)
# ══════════════════════════════════════════════
import tensorflow as tf
tf.random.set_seed(42)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Dropout, BatchNormalization,   # capas densas
    Conv2D, MaxPooling2D, Flatten         # capas convolucionales
)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

print(f"✅ TensorFlow {tf.__version__} OK")


In [ ]:
# ══════════════════════════════════════════════
#  IMPORTS SKLEARN — modelos clásicos + métricas
# ══════════════════════════════════════════════
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Métricas
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    confusion_matrix, classification_report,
    f1_score, recall_score, precision_score,
    roc_auc_score, roc_curve,
    average_precision_score
)

print("✅ Sklearn OK")


In [ ]:
# ══════════════════════════════════════════════
#  IMPORTS PYTORCH
# ══════════════════════════════════════════════
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ PyTorch {torch.__version__} — Dispositivo: {device}")


In [ ]:
# ══════════════════════════════════════════════
#  IMPORTS VISUALIZACIÓN AVANZADA (plotly)
# ══════════════════════════════════════════════
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("✅ Plotly OK")


---
# 🔍 SECCIÓN 2 — EDA & PREPROCESADO (Datos Tabulares)

## 2.1 Inspección inicial — siempre lo primero

In [ ]:
# ── CARGAR CSV ──────────────────────────────────────────────────────
df = pd.read_csv('tu_dataset.csv')

# ── INSPECCIÓN BÁSICA ────────────────────────────────────────────────
print(f"Shape: {df.shape[0]} filas × {df.shape[1]} columnas")
print()
df.head()


In [ ]:
df.info()          # tipos de datos, nulls


In [ ]:
df.describe()      # estadísticas numéricas


In [ ]:
# ── NULLS Y DUPLICADOS ──────────────────────────────────────────────
print("Valores nulos por columna:")
print(df.isnull().sum())
print()
print(f"Filas duplicadas: {df.duplicated().sum()}")


## 2.2 Visualización EDA

In [ ]:
# ── BOXPLOTS de numéricas vs target ─────────────────────────────────
# (Patrón del Proyecto Crédito)
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
n_cols = len(num_cols)
cols = 2
rows = math.ceil(n_cols / cols)

fig, axes = plt.subplots(rows, cols, figsize=(14, 5 * rows))
axes = axes.flatten()

for i, var in enumerate(num_cols):
    sns.boxplot(x='target_col', y=var, data=df, ax=axes[i], palette="Blues")
    axes[i].set_title(f'Distribución de {var}', fontweight='bold')

for j in range(i + 1, len(axes)):   # ocultar ejes sobrantes
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [ ]:
# ── DISTRIBUCIÓN DE LA VARIABLE OBJETIVO (pie chart plotly) ─────────
class_counts = df['target_col'].value_counts().reset_index()
class_counts.columns = ['Clase', 'Cantidad']

fig = px.pie(class_counts, values='Cantidad', names='Clase',
             title='Distribución Variable Objetivo',
             hole=0.4)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()
# ⚠️ Si hay desbalanceo → usar class_weight='balanced' + métricas robustas


In [ ]:
# ── MATRIZ DE CORRELACIÓN ───────────────────────────────────────────
plt.figure(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matriz de Correlación')
plt.show()
# Correlación > 0.7 entre features → posible multicolinealidad


## 2.3 Preprocesado completo — Datos Tabulares

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  PREPROCESADO COMPLETO (patrón FinalProjectUD4)
# ══════════════════════════════════════════════════════════════════
df_prep = df.copy()

# ── 1. Eliminar columnas irrelevantes (IDs, etc.) ────────────────
df_prep = df_prep.drop('id_column', axis=1)

# ── 2. Eliminar outliers imposibles ─────────────────────────────
df_prep = df_prep[df_prep['age'] <= 100]   # ejemplo: edades > 100

# ── 3. Imputar valores nulos ─────────────────────────────────────
# Moda (categóricas)
moda = df_prep['col_categorica'].mode()[0]
df_prep['col_categorica'] = df_prep['col_categorica'].fillna(moda)
# Media (numéricas)
# df_prep['col_numerica'] = df_prep['col_numerica'].fillna(df_prep['col_numerica'].mean())

# ── 4. Codificar la variable objetivo ───────────────────────────
df_prep['class'] = df_prep['class'].map({'good': 1, 'bad': 0})
# Alternativa: LabelEncoder
# le = LabelEncoder(); df_prep['class'] = le.fit_transform(df_prep['class'])

# ── 5. Separar X e y ─────────────────────────────────────────────
X = df_prep.drop('class', axis=1)
y = df_prep['class'].values

# ── 6. Train/Test split ESTRATIFICADO ────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=seed,
    stratify=y          # ← mantiene proporción de clases en ambos sets
)

# ── 7. One-Hot Encoding SIN fuga de datos ────────────────────────
X_train = pd.get_dummies(X_train, drop_first=True)
X_test  = pd.get_dummies(X_test,  drop_first=True)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
# join='left' → test se alinea con train (evita columnas de más en test)

# ── 8. Escalado con StandardScaler ──────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit+transform en TRAIN
X_test_scaled  = scaler.transform(X_test)        # solo transform en TEST
# ⚠️ NUNCA hacer fit sobre test → fuga de información

print(f"X_train: {X_train_scaled.shape} | X_test: {X_test_scaled.shape}")
print(f"Proporción clase positiva — Train: {y_train.mean():.2f} | Test: {y_test.mean():.2f}")


## 2.4 Preprocesado — Imágenes (MNIST / Fashion-MNIST)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  PREPROCESADO IMÁGENES KERAS
# ══════════════════════════════════════════════════════════════════
from keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()
# Shapes: X_train (60000, 28, 28)  y_train (60000,)

# ── Para MLP (Dense): aplanar ────────────────────────────────────
X_train_mlp = X_train.astype("float32") / 255          # normalizar
X_test_mlp  = X_test.astype("float32")  / 255
X_train_mlp = X_train_mlp.reshape(X_train_mlp.shape[0], 784)  # → (60000, 784)
X_test_mlp  = X_test_mlp.reshape(X_test_mlp.shape[0],   784)

# ── Para CNN: añadir canal ──────────────────────────────────────
X_train_cnn = X_train.astype("float32").reshape(-1, 28, 28, 1) / 255  # → (60000,28,28,1)
X_test_cnn  = X_test.astype("float32").reshape(-1, 28, 28, 1)  / 255
# -1 = Keras calcula automáticamente el nº de muestras

# ── One-hot encoding de labels (para categorical_crossentropy) ──
labels = np.unique(y_train)    # [0,1,2,...,9]
y_train_cat = to_categorical(y_train, len(labels))  # → (60000, 10)
y_test_cat  = to_categorical(y_test,  len(labels))  # → (10000, 10)

print("MLP shapes:", X_train_mlp.shape, y_train_cat.shape)
print("CNN shapes:", X_train_cnn.shape, y_train_cat.shape)


---
# 🔧 SECCIÓN 3 — Pipeline Keras — Datos Tabulares (Clasificación Binaria)

## 3.1 Modelo Simple (1 capa oculta)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  MODELO 1: RED SIMPLE (patrón FinalProjectUD4)
# ══════════════════════════════════════════════════════════════════
tf.keras.backend.clear_session()   # borra memoria basura de Keras

n_features = X_train_scaled.shape[1]   # nº de features tras One-Hot

model_1 = Sequential(name="Red_Simple")
# Capa oculta: 32 neuronas, ReLU
model_1.add(Dense(32, activation='relu', input_shape=(n_features,)))
# Params: (n_features × 32) + 32 = ?
# Capa salida: 1 neurona, sigmoid → clasificación binaria
model_1.add(Dense(1, activation='sigmoid'))
# Params: (32 × 1) + 1 = 33

model_1.compile(
    optimizer='adam',
    loss='binary_crossentropy',     # binaria → binary_crossentropy
    metrics=['accuracy']
)
model_1.summary()


## 3.2 Modelo Complejo (múltiples capas + Dropout)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  MODELO 2: RED COMPLEJA (patrón FinalProjectUD4)
# ══════════════════════════════════════════════════════════════════
model_2 = Sequential(name="Red_Compleja")

# Bloque 1: 64 neuronas + Dropout 30%
model_2.add(Dense(64, activation='relu', input_shape=(n_features,)))
model_2.add(Dropout(0.3))   # desactiva 30% neuronas en train

# Bloque 2: 32 neuronas + Dropout 20%
model_2.add(Dense(32, activation='relu'))
model_2.add(Dropout(0.2))

# Salida binaria
model_2.add(Dense(1, activation='sigmoid'))

# Justificación arquitectura 64→32→1:
#   - 64: captura patrones complejos
#   - 32: comprime la representación
#   - 1 + sigmoid: probabilidad de clase positiva

model_2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# Alternativa LR bajo para datasets desbalanceados:
# optimizer_suave = tf.keras.optimizers.Adam(learning_rate=0.0005)

model_2.summary()


## 3.3 Entrenamiento con EarlyStopping y class_weight

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  EARLYSTOPPING — evita overfitting en datasets pequeños
# ══════════════════════════════════════════════════════════════════
early_stop = EarlyStopping(
    monitor='val_loss',         # vigila la pérdida de validación
    patience=25,                # para si no mejora en 25 épocas
    restore_best_weights=True   # recupera los mejores pesos automáticamente
)
# ⚠️ Sin restore_best_weights=True el modelo se queda en los pesos del último epoch

# ── CLASS WEIGHT — corrige desbalanceo de clases ─────────────────
class_weight_values = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(zip(np.unique(y_train), class_weight_values))
print(f"Pesos de clase: {class_weight_dict}")
# Ejemplo: {0: 1.67, 1: 0.71} → da más importancia a la clase minoritaria

# ── ENTRENAMIENTO ────────────────────────────────────────────────
history_1 = model_1.fit(
    X_train_scaled, y_train,
    epochs=150,                     # máximo (EarlyStopping lo corta antes)
    batch_size=32,
    validation_split=0.2,           # 20% del train → validación
    callbacks=[early_stop],
    class_weight=class_weight_dict,
    verbose=0                       # silencioso
)
print(f"Modelo 1 detenido en epoch {len(history_1.epoch)}")

history_2 = model_2.fit(
    X_train_scaled, y_train,
    epochs=150, batch_size=32,
    validation_split=0.2,
    callbacks=[EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True)],
    class_weight=class_weight_dict,
    verbose=0
)
print(f"Modelo 2 detenido en epoch {len(history_2.epoch)}")


## 3.4 Evaluación binaria completa

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  EVALUACIÓN COMPLETA — clasificación binaria desbalanceada
# ══════════════════════════════════════════════════════════════════

# Probabilidades brutas (0.0 → 1.0)
y_prob_1 = model_1.predict(X_test_scaled, verbose=0).ravel()
y_prob_2 = model_2.predict(X_test_scaled, verbose=0).ravel()

# Clases predichas (umbral 0.5)
y_pred_1 = (y_prob_1 >= 0.5).astype(int)
y_pred_2 = (y_prob_2 >= 0.5).astype(int)

# ── Métricas para cada modelo ────────────────────────────────────
for name, y_prob, y_pred in [
    ("Modelo 1 (Simple)",  y_prob_1, y_pred_1),
    ("Modelo 2 (Complejo)", y_prob_2, y_pred_2)
]:
    loss, acc = model_1.evaluate(X_test_scaled, y_test, verbose=0)
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  Accuracy Global:    {accuracy_score(y_test, y_pred):.4f}")
    print(f"  Balanced Accuracy:  {balanced_accuracy_score(y_test, y_pred):.4f}")
    print(f"  ROC-AUC:            {roc_auc_score(y_test, y_prob):.4f}")
    print(f"  F1 (clase 0=bad):   {f1_score(y_test, y_pred, pos_label=0):.4f}")
    print(f"  Recall (clase 0):   {recall_score(y_test, y_pred, pos_label=0):.4f}")
    print()
    print(classification_report(y_test, y_pred, digits=4))

# ── CUÁNDO USAR CADA MÉTRICA ─────────────────────────────────────
# Accuracy       → solo si clases balanceadas
# Balanced Acc   → datasets desbalanceados (media de recalls por clase)
# ROC-AUC        → capacidad discriminativa global (mejor: 1.0)
# Recall clase 0 → cuántos malos pagadores detecta (crítico en fraude/crédito)
# F1             → equilibrio entre precisión y recall


---
# 🖼️ SECCIÓN 4 — Pipeline Keras — Imágenes (CNN)

## 4.1 CNN Base (sin regularización)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CNN BASE — patrón Ejemplo_RRNN_convolucionales
# ══════════════════════════════════════════════════════════════════
# Cálculo de shapes paso a paso:
#   Input:           (None, 28, 28, 1)
#   Conv2D(32,(5,5)): output = 28-5+1 = 24   → (None, 24, 24, 32)  params=832
#   MaxPool(2,2):     output = 24/2   = 12   → (None, 12, 12, 32)  params=0
#   Conv2D(64,(5,5)): output = 12-5+1 = 8    → (None,  8,  8, 64)  params=51264
#   MaxPool(2,2):     output = 8/2    = 4    → (None,  4,  4, 64)  params=0
#   Flatten:          4×4×64 = 1024          → (None, 1024)         params=0
#   Dense(10):                               → (None, 10)           params=10250

model_cnn = Sequential()
model_cnn.add(Conv2D(32, (5,5), activation='relu', input_shape=(28,28,1)))
model_cnn.add(MaxPooling2D((2,2)))
model_cnn.add(Conv2D(64, (5,5), activation='relu'))
model_cnn.add(MaxPooling2D((2,2)))
model_cnn.add(Flatten())
model_cnn.add(Dense(10, activation='softmax'))   # 10 clases → softmax

model_cnn.compile(
    loss='categorical_crossentropy',   # labels en one-hot
    optimizer='sgd',
    metrics=['accuracy']
)
model_cnn.summary()


## 4.2 CNN Avanzada (BatchNorm + Dropout) — Modelo 4

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CNN COMPLEJA — patrón Modelo 4 Fashion-MNIST
#  Secuencia estándar: Conv → BatchNorm → Dropout → MaxPool
# ══════════════════════════════════════════════════════════════════
def make_model():
    model = Sequential()

    # ── Bloque Conv 1 ──────────────────────────────────────────
    model.add(Conv2D(32, (3,3), activation='relu', strides=1,
                     padding='same', input_shape=(28,28,1)))
    model.add(BatchNormalization())                # normaliza activaciones
    model.add(Conv2D(32, (3,3), activation='relu', strides=1, padding='same'))
    model.add(BatchNormalization())
    model.add(Dropout(0.25))                       # 25% en capas conv

    # ── Bloque Conv 2 ──────────────────────────────────────────
    model.add(Conv2D(64, (3,3), activation='relu', strides=1, padding='same'))
    model.add(MaxPooling2D(pool_size=(2,2)))
    model.add(Dropout(0.25))

    # ── Bloque Conv 3 ──────────────────────────────────────────
    model.add(Conv2D(128, (3,3), activation='relu', strides=1, padding='same'))
    model.add(BatchNormalization())
    model.add(Dropout(0.25))

    # ── Clasificador ────────────────────────────────────────────
    model.add(Flatten())
    model.add(Dense(512, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))                        # 50% en capas dense
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))
    model.add(Dense(10, activation='softmax'))

    return model

model = make_model()
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
# sparse_categorical_crossentropy → labels como enteros (sin to_categorical)
model.summary()


## 4.3 LR Scheduler + Entrenamiento CNN

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  LR SCHEDULER — reduce learning rate exponencialmente
#  Epoch 0: lr=0.001  |  Epoch 5: lr≈0.00059  |  Epoch 10: lr≈0.00035
# ══════════════════════════════════════════════════════════════════
reduce_lr = tf.keras.callbacks.LearningRateScheduler(
    lambda x: 1e-3 * 0.9 ** x
)
# lambda x: → x es el número de epoch actual
# 0.9**x → decaimiento exponencial

model = make_model()
optimizer = tf.keras.optimizers.Adam(lr=0.001)
model.compile(optimizer=optimizer,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(
    train_images, train_labels,
    epochs=30,
    callbacks=[reduce_lr]
)
test_loss, test_acc = model.evaluate(test_images, test_labels)
print(f'\nTest accuracy: {test_acc:.4f}')


---
# 📊 SECCIÓN 5 — Métricas & Evaluación Completa

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TABLA DE MÉTRICAS — cuándo usar cada una
# ══════════════════════════════════════════════════════════════════
# accuracy           → clases balanceadas. Engaña en desbalanceo.
# balanced_accuracy  → promedio de recall por clase. Robusta al desbalanceo.
# ROC-AUC            → capacidad de separar clases (0.5=azar, 1.0=perfecto)
# F1-score           → media armónica de precision y recall. Buena con desbalanceo.
# Recall             → de todos los positivos reales, cuántos detecta. CRÍTICO en fraude.
# Precision          → de los que predijo positivo, cuántos son realmente positivos.

# ── Evaluación sklearn ───────────────────────────────────────────
y_prob = model.predict(X_test_scaled).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print(f"Accuracy:          {accuracy_score(y_test, y_pred):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:           {roc_auc_score(y_test, y_prob):.4f}")
print(f"F1 (pos=0):        {f1_score(y_test, y_pred, pos_label=0):.4f}")
print(f"Recall (pos=0):    {recall_score(y_test, y_pred, pos_label=0):.4f}")
print()
print(classification_report(y_test, y_pred, digits=4))
# classification_report incluye: precision, recall, f1-score, support por clase


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  INTERPRETAR classification_report
# ══════════════════════════════════════════════════════════════════
# Ejemplo de salida:
#               precision  recall  f1-score  support
#            0      0.67    0.67      0.67       60    ← clase mala (bad)
#            1      0.84    0.84      0.84      140    ← clase buena (good)
#     accuracy                        0.78      200
#    macro avg      0.76    0.76      0.76      200    ← promedio sin ponderar
# weighted avg      0.78    0.78      0.78      200    ← promedio ponderado por support

# ── Interpretación rápida ─────────────────────────────────────────
# Recall clase 0 = 0.67 → detecta 2 de cada 3 malos pagadores
# Recall clase 1 = 0.84 → detecta 84% de buenos pagadores
# Si recall clase minoritaria < 0.50 → modelo inútil para detectar riesgo
# Si balanced_accuracy < 0.60 → peor que adivinar aleatoriamente con pesos

print("✅ Ver tabla de arriba para interpretar")


---
# 📈 SECCIÓN 6 — Gráficas de Entrenamiento e Interpretación

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FUNCIÓN UNIVERSAL DE PLOTS DE ENTRENAMIENTO
# ══════════════════════════════════════════════════════════════════
def plot_training(history, title="Modelo"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # ── Accuracy ──────────────────────────────────────────────────
    axes[0].plot(history.history['accuracy'],     label='Train', color='blue')
    axes[0].plot(history.history['val_accuracy'], label='Validación', color='orange', linestyle='--')
    axes[0].set_title(f'{title}: Accuracy', fontweight='bold')
    axes[0].set_xlabel('Epochs'); axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # ── Loss ──────────────────────────────────────────────────────
    axes[1].plot(history.history['loss'],     label='Train', color='blue')
    axes[1].plot(history.history['val_loss'], label='Validación', color='orange', linestyle='--')
    axes[1].set_title(f'{title}: Loss', fontweight='bold')
    axes[1].set_xlabel('Epochs'); axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # ── Diagnóstico automático ────────────────────────────────────
    final_train_loss = history.history['loss'][-1]
    final_val_loss   = history.history['val_loss'][-1]
    gap = final_val_loss - final_train_loss
    
    if gap > 0.1:
        print("⚠️  POSIBLE OVERFITTING — val_loss mucho mayor que train_loss")
        print("    → Añadir Dropout, reducir epochs, más datos")
    elif final_train_loss > 0.5:
        print("⚠️  POSIBLE UNDERFITTING — loss alta")
        print("    → Más neuronas, más epochs, menos Dropout, bajar lr")
    else:
        print("✅ Entrenamiento normal — curvas convergentes")

# Uso:
# plot_training(history_1, "Modelo Simple")
# plot_training(history_2, "Modelo Complejo")


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  COMPARATIVA DOS MODELOS (2×2 grid)
# ══════════════════════════════════════════════════════════════════
def plot_comparativa_2modelos(h1, h2, nombre1="Modelo 1", nombre2="Modelo 2"):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Loss Modelo 1
    axes[0,0].plot(h1.history['loss'],     label='Train', color='blue')
    axes[0,0].plot(h1.history['val_loss'], label='Val', color='orange', linestyle='--')
    axes[0,0].set_title(f'{nombre1}: Loss', fontweight='bold'); axes[0,0].legend()

    # Loss Modelo 2
    axes[0,1].plot(h2.history['loss'],     label='Train', color='blue')
    axes[0,1].plot(h2.history['val_loss'], label='Val', color='orange', linestyle='--')
    axes[0,1].set_title(f'{nombre2}: Loss', fontweight='bold'); axes[0,1].legend()

    # Confusion matrix Modelo 1
    sns.heatmap(confusion_matrix(y_test, y_pred_1), annot=True, fmt='d',
                cmap='Blues', ax=axes[1,0])
    axes[1,0].set_title(f'Conf. Matrix: {nombre1}', fontweight='bold')
    axes[1,0].set_ylabel('Real'); axes[1,0].set_xlabel('Predicho')

    # Confusion matrix Modelo 2
    sns.heatmap(confusion_matrix(y_test, y_pred_2), annot=True, fmt='d',
                cmap='Greens', ax=axes[1,1])
    axes[1,1].set_title(f'Conf. Matrix: {nombre2}', fontweight='bold')
    axes[1,1].set_ylabel('Real'); axes[1,1].set_xlabel('Predicho')

    plt.tight_layout(); plt.show()

# Uso:
# plot_comparativa_2modelos(history_1, history_2, "Simple", "Complejo")


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CURVA ROC (muy preguntada en examen)
# ══════════════════════════════════════════════════════════════════
def plot_roc(y_test, probs_dict, title="Curva ROC"):
    # probs_dict = {"Nombre modelo": y_prob_array}
    plt.figure(figsize=(8, 6))
    plt.plot([0,1],[0,1], 'k--', label='Azar (AUC=0.5)')
    
    for name, y_prob in probs_dict.items():
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc = roc_auc_score(y_test, y_prob)
        plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
    
    plt.xlabel('Tasa Falsos Positivos (FPR)')
    plt.ylabel('Tasa Verdaderos Positivos (TPR / Recall)')
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
    # AUC=1.0 → perfecto | AUC=0.5 → inútil | AUC>0.7 → aceptable

# Uso:
# plot_roc(y_test, {"Modelo 1": y_prob_1, "Modelo 2": y_prob_2, "LR": y_prob_lr})


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  GRÁFICA COMPARATIVA DE MÉTRICAS (barras)
# ══════════════════════════════════════════════════════════════════
def plot_metricas_comparativa(metricas_dict):
    # metricas_dict = {
    #     'Métrica': ['Accuracy', 'Balanced Acc', 'ROC-AUC'],
    #     'Modelo 1': [0.78, 0.68, 0.74],
    #     'Modelo 2': [0.80, 0.70, 0.78]
    # }
    df_comp = pd.DataFrame(metricas_dict).set_index('Métrica')
    ax = df_comp.plot(kind='bar', figsize=(12, 6), width=0.75, alpha=0.85)
    
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.3f}",
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    plt.ylabel('Score (0 → 1)'); plt.xticks(rotation=0)
    plt.ylim(0, 1.05); plt.grid(axis='y', alpha=0.4)
    plt.title('Comparación de Modelos'); plt.legend(loc='lower right')
    plt.tight_layout(); plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  INTERPRETAR GRÁFICAS — CHEATSHEET VISUAL
# ══════════════════════════════════════════════════════════════════
interpretacion = {
    "Situación": [
        "Curvas train y val juntas y mejorando",
        "val_loss SUBE, train_loss BAJA",
        "Ambas curvas altas desde el inicio",
        "Curvas con grandes oscilaciones",
        "Loss no baja desde epoch 1"
    ],
    "Diagnóstico": [
        "✅ Entrenamiento correcto",
        "⚠️ OVERFITTING — memoriza train",
        "⚠️ UNDERFITTING — modelo insuficiente",
        "⚠️ Entrenamiento INESTABLE",
        "❌ Problema de datos o LR"
    ],
    "Solución": [
        "Nada que cambiar",
        "Dropout, menos epochs, más datos, EarlyStopping",
        "Más neuronas/capas, más epochs, menos Dropout",
        "Reducir learning rate, LR scheduler",
        "Normalizar datos, ajustar LR, revisar labels"
    ]
}
pd.DataFrame(interpretacion)


---
# 🤖 SECCIÓN 7 — Modelos Sklearn + Cross-Validation

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  COMPARAR MODELOS CON STRATIFIED K-FOLD (patrón FinalProjectUD4)
# ══════════════════════════════════════════════════════════════════
models = [
    ('LR',   LogisticRegression(max_iter=1000, class_weight='balanced')),
    ('KNN',  KNeighborsClassifier()),
    ('CART', DecisionTreeClassifier(random_state=seed)),
    ('RF',   RandomForestClassifier(random_state=seed, class_weight='balanced')),
    ('SVM',  SVC(gamma='auto', class_weight='balanced'))
]

results, names = [], []
scoring = 'balanced_accuracy'   # métrica robusta al desbalanceo

print("━"*60)
print("  CROSS-VALIDATION (10-Fold Estratificado)")
print("━"*60)
for name, model in models:
    kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
    cv_results = cross_val_score(model, X_train_scaled, y_train,
                                  cv=kfold, scoring=scoring)
    results.append(cv_results)
    names.append(name)
    print(f"  {name:6s} → Media: {cv_results.mean():.4f} | Std: {cv_results.std():.4f}")

# ── Boxplot comparativo ──────────────────────────────────────────
plt.figure(figsize=(10, 6))
palette = sns.color_palette('Set2', n_colors=len(names))
box = plt.boxplot(results, patch_artist=True, labels=names,
                  showmeans=True,
                  meanprops=dict(marker='o', markerfacecolor='white',
                                 markeredgecolor='black', markersize=6),
                  medianprops=dict(color='black', linewidth=2))
for patch, color in zip(box['boxes'], palette):
    patch.set_facecolor(color); patch.set_alpha(0.8)

plt.title('Comparación de Modelos (Stratified 10-Fold CV)', fontsize=14, fontweight='bold')
plt.ylabel('Balanced Accuracy')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  ENTRENAR Y EVALUAR MODELO SKLEARN GANADOR
# ══════════════════════════════════════════════════════════════════
# Tras la CV, entrena el mejor modelo en todo el train set

lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=seed)
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]
# predict_proba → probabilidades | [:,1] = prob de clase positiva

print(f"Accuracy:          {accuracy_score(y_test, y_pred_lr)*100:.2f}%")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_lr):.4f}")
print(f"ROC-AUC:           {roc_auc_score(y_test, y_prob_lr):.4f}")
print()
print(classification_report(y_test, y_pred_lr))

plt.figure(figsize=(6,5))
sns.heatmap(confusion_matrix(y_test, y_pred_lr), annot=True, fmt='d',
            cmap='Blues', annot_kws={"size":14})
plt.title('Confusion Matrix: Regresión Logística', fontweight='bold')
plt.ylabel('Real'); plt.xlabel('Predicho')
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  LEER CONFUSION MATRIX — cómo interpretar los 4 valores
# ══════════════════════════════════════════════════════════════════
#
#              Predicho
#              0 (bad)  1 (good)
#  Real 0 (bad)  [ TN ]  [ FP ]   FP = predijo bueno pero era malo
#  Real 1 (good) [ FN ]  [ TP ]   FN = predijo malo pero era bueno
#
#  TN (True Negative):  malo detectado correctamente como malo   ← CRÍTICO en crédito
#  TP (True Positive):  bueno detectado correctamente como bueno
#  FP (False Positive): falsa alarma (rechazamos a un buen cliente)
#  FN (False Negative): error grave (damos crédito a un mal pagador)
#
#  En crédito/fraude: FN >> FP (un FN puede costar mucho dinero)
#  Por eso: maximizar Recall de clase 0 (= minimizar FN)

print("✅ Interpretación de confusion matrix OK")


---
# 🔥 SECCIÓN 8 — PyTorch: Módulos, DataLoaders y Bucle de Entrenamiento

## 8.1 Definir modelos — Module vs Sequential

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  PYTORCH: DOS FORMAS DE DEFINIR MODELOS
# ══════════════════════════════════════════════════════════════════

# ── Forma 1: nn.Module (más flexible, permite forward personalizado) ──
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.flatten = nn.Flatten()      # aplana imagen: 28×28 → 784
        self.fc1 = nn.Linear(784, 128)   # capa lineal (equiv. Dense en Keras)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)    # salida: 10 clases

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

modelo_clase = Net()
print("Modelo con clase:")
print(modelo_clase)
print()

# ── Forma 2: nn.Sequential (más simple, arquitectura lineal) ────────
modelo_seq = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)
print("Modelo Sequential:")
print(modelo_seq)


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  PYTORCH: CNN COMPLETA (patrón training.ipynb)
# ══════════════════════════════════════════════════════════════════
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        self.conv1    = nn.Conv2d(1, 32, 3, 1)    # (ch_in, ch_out, kernel, stride)
        self.conv2    = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout2d(0.25)
        self.dropout2 = nn.Dropout2d(0.5)
        self.fc1      = nn.Linear(9216, 128)       # 9216 = 12×12×64 (tras conv+pool)
        self.fc2      = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)          # MaxPool 2×2
        x = self.dropout1(x)
        x = torch.flatten(x, 1)         # equivale a Flatten()
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)  # log_softmax + nll_loss = crossentropy
        return output

modelo_cnn = ConvNet()
print(modelo_cnn)


## 8.2 DataLoader — cargar datos por batches

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  PYTORCH: DataLoader completo con transformaciones
# ══════════════════════════════════════════════════════════════════
transform = transforms.Compose([
    transforms.ToTensor(),                          # PIL → tensor [0,1]
    transforms.Normalize((0.1307,), (0.3081,))      # media, std de MNIST
])

# Carga del dataset
train_dataset = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST('./data', train=False, download=True, transform=transform)

# DataLoader: divide en batches y mezcla
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=1000, shuffle=False)
# shuffle=True en train → evita que el modelo aprenda el orden de los datos
# shuffle=False en test → no importa el orden para evaluar

print(f"Batches en train: {len(train_loader)} ({len(train_dataset)} muestras ÷ 64)")
print(f"Batches en test:  {len(test_loader)}")


## 8.3 Bucle de entrenamiento PyTorch

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  PYTORCH: BUCLE DE ENTRENAMIENTO ESTÁNDAR
#  Los 6 pasos dentro de cada batch:
#  1. Cargar datos → 2. Forward → 3. Loss → 4. Zero grad → 5. Backward → 6. Step
# ══════════════════════════════════════════════════════════════════
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

modelo = ConvNet().to(device)
optimizer = optim.SGD(modelo.parameters(), lr=0.01, momentum=0.5)
# Alternativa Adam: optim.Adam(modelo.parameters(), lr=0.001)

def train_epoch(model, loader, optimizer, epoch):
    model.train()   # ← activa Dropout y BatchNorm en modo entrenamiento
    running_loss = 0.0
    
    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        
        # PASO 4: Zero grad (siempre ANTES del forward)
        optimizer.zero_grad()
        
        # PASO 2: Forward pass
        output = model(data)
        
        # PASO 3: Calcular loss
        loss = F.nll_loss(output, target)
        # Alternativa: nn.CrossEntropyLoss()(output, target)
        
        # PASO 5: Backward (calcula gradientes)
        loss.backward()
        
        # PASO 6: Actualizar pesos
        optimizer.step()
        
        running_loss += loss.item()
        if batch_idx % 100 == 0:
            print(f'Epoch {epoch} [{batch_idx*len(data)}/{len(loader.dataset)}] '
                  f'Loss: {loss.item():.4f}')
    
    return running_loss / len(loader)


def evaluate(model, loader):
    model.eval()    # ← desactiva Dropout y BatchNorm en modo evaluación
    correct = 0; total = 0; test_loss = 0
    
    with torch.no_grad():   # ← no calcular gradientes en evaluación
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()
            pred = output.argmax(dim=1, keepdim=True)   # clase con mayor prob
            correct += pred.eq(target.view_as(pred)).sum().item()
            total += target.size(0)
    
    accuracy = 100. * correct / total
    avg_loss = test_loss / total
    print(f'Test → Loss: {avg_loss:.4f} | Accuracy: {correct}/{total} ({accuracy:.1f}%)')
    return accuracy


# ── Bucle principal ───────────────────────────────────────────────
for epoch in range(1, 6):
    train_epoch(modelo, train_loader, optimizer, epoch)
    evaluate(modelo, test_loader)


## 8.4 Visualizar arquitectura PyTorch

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  VISUALIZAR MODELO PYTORCH (equivalente a model.summary() de Keras)
# ══════════════════════════════════════════════════════════════════

# Opción 1: torchsummary (pip install torchsummary)
# from torchsummary import summary
# model.to(device)
# summary(model, input_size=(1, 28, 28))  # (canales, alto, ancho)

# Opción 2: recorrer parámetros manualmente
print("═"*50)
print("  Parámetros del modelo")
print("═"*50)
total_params = 0
for name, param in modelo.named_parameters():
    params = param.numel()
    total_params += params
    print(f"  {name:30s} {str(list(param.shape)):20s} {params:>8,} params")
print("─"*50)
print(f"  {'TOTAL':30s} {'':20s} {total_params:>8,} params")


## 8.5 CNN para CIFAR-10 (patrón ejercicio)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CNN CIFAR-10 (patrón training.ipynb ejercicio completo)
#  CIFAR-10: imágenes 32×32 RGB (3 canales), 10 clases
# ══════════════════════════════════════════════════════════════════
import torchvision

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # 3 canales RGB
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                         download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=4, shuffle=True, num_workers=2)
testloader  = DataLoader(testset,  batch_size=4, shuffle=False, num_workers=2)

classes = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')

class CifarNet(nn.Module):
    def __init__(self):
        super(CifarNet, self).__init__()
        # Input: (3, 32, 32)
        self.conv1 = nn.Conv2d(3, 6, 5)      # → (6, 28, 28)
        self.pool  = nn.MaxPool2d(2, 2)       # → (6, 14, 14)
        self.conv2 = nn.Conv2d(6, 16, 5)     # → (16, 10, 10) → pool → (16, 5, 5)
        self.fc1   = nn.Linear(16 * 5 * 5, 120)
        self.fc2   = nn.Linear(120, 84)
        self.fc3   = nn.Linear(84, 10)       # 10 clases

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)          # aplanar: equiv. a Flatten()
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

net = CifarNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

print("Arquitectura CIFAR-10:")
print(net)


---
# 🧮 SECCIÓN 9 — Cálculo de Parámetros (MUY preguntado en examen)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FÓRMULAS DE PARÁMETROS
# ══════════════════════════════════════════════════════════════════

# ── DENSE / Linear ────────────────────────────────────────────────
# params = (entradas × neuronas) + neuronas_bias
# Dense(32, input_dim=48): (48×32) + 32 = 1,568
# Dense(1,  anterior=32):  (32×1)  + 1  = 33
# Dense(64, input_dim=48): (48×64) + 64 = 3,136
# Dense(10, anterior=1024):(1024×10)+10 = 10,250

# ── CONV2D ────────────────────────────────────────────────────────
# params = ((kernel_w × kernel_h × ch_entrada) + 1) × num_filtros
# Conv2D(32,(5,5), ch_in=1):   ((5×5×1)+1)×32   = 832
# Conv2D(64,(5,5), ch_in=32):  ((5×5×32)+1)×64  = 51,264
# Conv2D(32,(3,3), ch_in=1):   ((3×3×1)+1)×32   = 320
# Conv2D(6, (5,5), ch_in=3):   ((5×5×3)+1)×6    = 456

# ── OUTPUT SHAPE CONV2D ───────────────────────────────────────────
# padding='valid': output = input - kernel + 1
#   28 - 5 + 1 = 24  |  24 - 5 + 1 = 20  |  28 - 3 + 1 = 26
# padding='same':  output = input (no cambia)

# ── MAXPOOLING ────────────────────────────────────────────────────
# output = input ÷ pool_size
# 24 ÷ 2 = 12  |  12 ÷ 2 = 6  |  8 ÷ 2 = 4
# params = SIEMPRE 0

# ── FLATTEN ───────────────────────────────────────────────────────
# output = alto × ancho × canales
# (4, 4, 64) → 1024   |   (12, 12, 32) → 4608

def calcular_params_dense(entradas, neuronas):
    return (entradas * neuronas) + neuronas

def calcular_params_conv(kernel_h, kernel_w, ch_entrada, num_filtros):
    return ((kernel_h * kernel_w * ch_entrada) + 1) * num_filtros

def calcular_output_conv(entrada, kernel, padding='valid'):
    if padding == 'valid':
        return entrada - kernel + 1
    else:  # 'same'
        return entrada

def calcular_output_pool(entrada, pool_size=2):
    return entrada // pool_size

# ── Ejemplo: Red Proyecto Crédito (n_features=48) ──────────────────
n = 48
print("=== Modelo 1: Red Simple (n_features=48) ===")
p1 = calcular_params_dense(n,  32); print(f"Dense(32, input={n}): {p1:,}")
p2 = calcular_params_dense(32,  1); print(f"Dense(1):             {p2:,}")
print(f"TOTAL: {p1+p2:,}")

print()
print("=== Modelo 2: Red Compleja ===")
q1 = calcular_params_dense(n,  64); print(f"Dense(64, input={n}): {q1:,}")
q2 = calcular_params_dense(64, 32); print(f"Dense(32):            {q2:,}")
q3 = calcular_params_dense(32,  1); print(f"Dense(1):             {q3:,}")
print(f"TOTAL: {q1+q2+q3:,}")

print()
print("=== CNN MNIST (28×28×1) ===")
c1 = calcular_params_conv(5,5,1,32);  out1 = calcular_output_conv(28,5)
p1 = calcular_output_pool(out1)
c2 = calcular_params_conv(5,5,32,64); out2 = calcular_output_conv(p1,5)
p2 = calcular_output_pool(out2)
fl = p2*p2*64
d1 = calcular_params_dense(fl, 10)
print(f"Conv2D(32,(5,5)):  {c1:,}   → output: ({out1}×{out1}×32)")
print(f"MaxPool(2,2):      0       → output: ({p1}×{p1}×32)")
print(f"Conv2D(64,(5,5)):  {c2:,}  → output: ({out2}×{out2}×64)")
print(f"MaxPool(2,2):      0       → output: ({p2}×{p2}×64)")
print(f"Flatten:           0       → output: ({fl},)")
print(f"Dense(10):         {d1:,}  → output: (10,)")
print(f"TOTAL: {c1+c2+d1:,}")


---
# 🐛 SECCIÓN 10 — Debugging y Errores Comunes

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TABLA DE ERRORES COMUNES Y SOLUCIONES
# ══════════════════════════════════════════════════════════════════
errores = {
    "Error / Síntoma": [
        "Loss = NaN desde epoch 1",
        "Accuracy ~10% desde inicio (multiclase)",
        "Accuracy ~50% desde inicio (binaria)",
        "ValueError: Input shape incompatible",
        "CNN: error de shape en Conv2D",
        "val_loss sube, train_loss baja",
        "Ambas curvas bajas y no mejoran",
        "Curvas con grandes oscilaciones",
        "to_categorical + sparse_categorical juntos",
        "scaler.fit en test set",
    ],
    "Causa": [
        "LR muy alto o datos sin normalizar",
        "Labels mal codificados o loss equivocada",
        "class_weight no aplicado en desbalanceo",
        "X shape no coincide con input_shape del modelo",
        "Imagen sin canal: (N,28,28) en vez de (N,28,28,1)",
        "Overfitting — modelo memoriza train",
        "Underfitting — modelo insuficiente",
        "Learning rate demasiado alto",
        "Inconsistencia loss/formato labels",
        "Fuga de información (data leakage)",
    ],
    "Solución": [
        "Normalizar /255, bajar lr ×10",
        "Verificar to_categorical y loss apropiada",
        "class_weight='balanced' o compute_class_weight",
        "Comprobar X.shape vs input_shape/input_dim",
        "reshape(-1, 28, 28, 1)",
        "EarlyStopping, Dropout, menos epochs",
        "Más neuronas/capas, más epochs, menos Dropout",
        "LR scheduler o bajar lr",
        "Usar solo categorical_crossentropy+to_categorical OR sparse+enteros",
        "fit solo en train, transform en train y test",
    ]
}
pd.DataFrame(errores)


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CHECKLIST PRE-ENTRENAMIENTO — ejecutar siempre
# ══════════════════════════════════════════════════════════════════
def checklist_preentrenamiento(X_train, X_test, y_train, y_test):
    print("🔍 CHECKLIST PRE-ENTRENAMIENTO")
    print("─"*50)
    
    # 1. Shapes
    print(f"X_train shape: {X_train.shape}  | dtype: {X_train.dtype}")
    print(f"X_test  shape: {X_test.shape}   | dtype: {X_test.dtype}")
    print(f"y_train shape: {y_train.shape}")
    print(f"y_test  shape: {y_test.shape}")
    
    # 2. Rango de valores
    print(f"X_train rango: [{X_train.min():.2f}, {X_train.max():.2f}]")
    if X_train.max() > 10:
        print("  ⚠️  Valores > 10 → ¿has normalizado?")
    else:
        print("  ✅ Rango normalizado")
    
    # 3. NaN
    nan_count = np.isnan(X_train).sum()
    print(f"NaN en X_train: {nan_count}", "✅" if nan_count==0 else "❌ ARREGLAR")
    
    # 4. Proporción de clases
    unique, counts = np.unique(y_train, return_counts=True)
    print(f"Clases: {dict(zip(unique, counts))}")
    if len(unique) == 2:
        ratio = max(counts) / min(counts)
        if ratio > 2:
            print(f"  ⚠️  Desbalanceo {ratio:.1f}:1 → usar class_weight='balanced'")
        else:
            print(f"  ✅ Clases balanceadas")
    
    print("─"*50)

# Uso:
# checklist_preentrenamiento(X_train_scaled, X_test_scaled, y_train, y_test)


---
# 📋 SECCIÓN 11 — Plantillas Rápidas (Copiar-Pegar en Examen)

## 11.1 Plantilla: Clasificación Binaria Tabulares (Proyecto Crédito)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  PLANTILLA COMPLETA — CLASIFICACIÓN BINARIA DATOS TABULARES
# ══════════════════════════════════════════════════════════════════
import os, warnings; os.environ['TF_CPP_MIN_LOG_LEVEL']='2'; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score,
                              classification_report, confusion_matrix)
seed = 42; np.random.seed(seed); tf.random.set_seed(seed)

# ── 1. CARGAR ──────────────────────────────────────────────────────
df = pd.read_csv('dataset.csv')
df = df.drop('id', axis=1)                           # eliminar ID
df['target'] = df['target'].map({'good':1, 'bad':0}) # codificar objetivo

# ── 2. PREPROCESAR ─────────────────────────────────────────────────
X = df.drop('target', axis=1)
y = df['target'].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=seed, stratify=y)
X_train = pd.get_dummies(X_train, drop_first=True)
X_test  = pd.get_dummies(X_test,  drop_first=True)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
n = X_train_s.shape[1]

# ── 3. CLASS WEIGHT ────────────────────────────────────────────────
cw = dict(zip(np.unique(y_train),
    compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)))

# ── 4. MODELO ──────────────────────────────────────────────────────
tf.keras.backend.clear_session()
model = Sequential([
    Dense(64, activation='relu', input_shape=(n,)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1,  activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# ── 5. ENTRENAR ────────────────────────────────────────────────────
history = model.fit(X_train_s, y_train, epochs=150, batch_size=32,
    validation_split=0.2, class_weight=cw, verbose=0,
    callbacks=[EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True)])
print(f"Detenido en epoch {len(history.epoch)}")

# ── 6. EVALUAR ─────────────────────────────────────────────────────
y_prob = model.predict(X_test_s, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:           {roc_auc_score(y_test, y_prob):.4f}")
print(classification_report(y_test, y_pred, digits=4))

# ── 7. GRÁFICAS ─────────────────────────────────────────────────────
fig, ax = plt.subplots(1, 2, figsize=(14,5))
ax[0].plot(history.history['loss'], label='Train'); ax[0].plot(history.history['val_loss'], label='Val', ls='--')
ax[0].set_title('Loss'); ax[0].legend()
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', ax=ax[1])
ax[1].set_title('Confusion Matrix'); ax[1].set_ylabel('Real'); ax[1].set_xlabel('Predicho')
plt.tight_layout(); plt.show()


## 11.2 Plantilla: CNN Imágenes (Fashion-MNIST)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  PLANTILLA COMPLETA — CNN IMÁGENES
# ══════════════════════════════════════════════════════════════════
import tensorflow as tf
import numpy as np, matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
tf.random.set_seed(42)

# ── 1. CARGAR Y PREPROCESAR ────────────────────────────────────────
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255
X_test  = X_test.reshape(-1, 28, 28, 1).astype('float32')  / 255
# y_train y_test → enteros 0-9 → usar sparse_categorical_crossentropy

# ── 2. MODELO ──────────────────────────────────────────────────────
def make_model():
    m = Sequential()
    m.add(Conv2D(32,(3,3), activation='relu', padding='same', input_shape=(28,28,1)))
    m.add(BatchNormalization()); m.add(Dropout(0.25))
    m.add(Conv2D(64,(3,3), activation='relu', padding='same'))
    m.add(MaxPooling2D(2,2)); m.add(Dropout(0.25))
    m.add(Conv2D(128,(3,3), activation='relu', padding='same'))
    m.add(BatchNormalization()); m.add(Dropout(0.25))
    m.add(Flatten())
    m.add(Dense(512, activation='relu')); m.add(BatchNormalization()); m.add(Dropout(0.5))
    m.add(Dense(10, activation='softmax'))
    return m

model = make_model()
reduce_lr = tf.keras.callbacks.LearningRateScheduler(lambda x: 1e-3 * 0.9 ** x)
model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# ── 3. ENTRENAR ────────────────────────────────────────────────────
history = model.fit(X_train, y_train, epochs=15, callbacks=[reduce_lr])

# ── 4. EVALUAR ─────────────────────────────────────────────────────
_, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {acc:.4f}")
predictions = model.predict(X_test)
print(f"Clase predicha imagen 0: {np.argmax(predictions[0])}")

# ── 5. GRÁFICAS ────────────────────────────────────────────────────
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy']); plt.title('Accuracy'); plt.xlabel('Epochs')
plt.subplot(1,2,2)
plt.plot(history.history['loss']); plt.title('Loss'); plt.xlabel('Epochs')
plt.tight_layout(); plt.show()


## 11.3 Plantilla: PyTorch MLP completo

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  PLANTILLA COMPLETA — PYTORCH MLP MNIST
# ══════════════════════════════════════════════════════════════════
import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── 1. DATOS ───────────────────────────────────────────────────────
transform = transforms.Compose([transforms.ToTensor(),
                                 transforms.Normalize((0.1307,),(0.3081,))])
train_loader = DataLoader(datasets.MNIST('./data',train=True,download=True,transform=transform),
                           batch_size=64, shuffle=True)
test_loader  = DataLoader(datasets.MNIST('./data',train=False,download=True,transform=transform),
                           batch_size=1000, shuffle=False)

# ── 2. MODELO ──────────────────────────────────────────────────────
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128), nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)

# ── 3. BUCLE ENTRENAMIENTO ─────────────────────────────────────────
for epoch in range(5):
    model.train()
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(data), target)
        loss.backward()
        optimizer.step()
    
    # Evaluación
    model.eval(); correct = 0; total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            pred = model(data).argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)
    print(f"Epoch {epoch+1}/5 — Accuracy: {100*correct/total:.2f}%")


## 11.4 Tabla de referencia rápida — Funciones clave

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TABLA RESUMEN — funciones más importantes
# ══════════════════════════════════════════════════════════════════
referencia = {
    "Función": [
        "model.compile()", "model.fit()", "model.evaluate()", "model.predict()",
        "model.summary()", "EarlyStopping()", "compute_class_weight()",
        "to_categorical()", "StandardScaler().fit_transform()", "pd.get_dummies()",
        "X.reshape(-1,28,28,1)", "np.argmax(pred, axis=1)",
        "(y_prob>=0.5).astype(int)", "train_test_split(...,stratify=y)",
        "cross_val_score()", "confusion_matrix()", "classification_report()",
        "roc_auc_score()", "balanced_accuracy_score()",
        "optimizer.zero_grad()", "loss.backward()", "optimizer.step()"
    ],
    "Para qué": [
        "Configura loss/optimizer/metrics", "Entrena el modelo",
        "Evalúa → devuelve [loss, metric]", "Genera predicciones (prob)",
        "Muestra arquitectura y params", "Corta si val_loss no mejora",
        "Calcula pesos para clases desbalanceadas", "Convierte labels a one-hot",
        "Normaliza (fit en train, transform en test)", "One-hot de categóricas (sin label leakage)",
        "Añade canal a imagen para CNN", "Clase con mayor probabilidad",
        "Probabilidad → clase 0/1", "Split manteniendo proporción de clases",
        "Validación cruzada múltiple", "Matriz de confusión",
        "Precision/recall/F1 por clase", "Área bajo curva ROC",
        "Accuracy media por clase (robusta a desbalanceo)",
        "Borra gradientes acumulados (PyTorch)", "Calcula gradientes (PyTorch)",
        "Actualiza pesos (PyTorch)"
    ],
    "Framework": [
        "Keras","Keras","Keras","Keras","Keras","Keras","Sklearn",
        "Keras","Sklearn","Pandas","NumPy","NumPy","NumPy","Sklearn",
        "Sklearn","Sklearn","Sklearn","Sklearn","Sklearn",
        "PyTorch","PyTorch","PyTorch"
    ]
}
pd.DataFrame(referencia)


## 11.5 Tabla de decisión final — qué usar en cada caso

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TABLA DE DECISIÓN — qué configurar según el problema
# ══════════════════════════════════════════════════════════════════
decision = {
    "Tipo de problema": [
        "Regresión", "Clasificación binaria", "Clasificación binaria",
        "Clasificación multiclase", "Clasificación multiclase"
    ],
    "Loss": [
        "mean_squared_error", "binary_crossentropy", "mean_squared_error",
        "categorical_crossentropy", "sparse_categorical_crossentropy"
    ],
    "Activación salida": [
        "lineal (ninguna)", "sigmoid", "sigmoid",
        "softmax", "softmax"
    ],
    "Neuronas salida": ["1", "1", "1", "N clases", "N clases"],
    "Formato labels y": [
        "float continuo", "0 ó 1", "0 ó 1",
        "one-hot (to_categorical)", "enteros 0,1,2..."
    ],
    "Métrica principal": [
        "mae / mse", "accuracy / ROC-AUC", "binary_accuracy",
        "accuracy", "accuracy"
    ]
}
pd.DataFrame(decision)


---
# 📐 SECCIÓN 12 — Fórmulas de Examen (Memorizar)

## Parámetros Dense/Linear
```
params = (entradas × neuronas) + neuronas
```
- Dense(32, input=48): **(48×32) + 32 = 1,568**
- Dense(1, anterior=32): **(32×1) + 1 = 33**

## Parámetros Conv2D
```
params = ((kernel_h × kernel_w × canales_entrada) + 1) × num_filtros
```
- Conv2D(32,(5,5), ch_in=1): **((5×5×1)+1)×32 = 832**
- Conv2D(64,(5,5), ch_in=32): **((5×5×32)+1)×64 = 51,264**

## Output Shape Conv2D
```
padding='valid':  output = input - kernel + 1   → 28-5+1 = 24
padding='same':   output = input                → 28 → 28
```

## Output Shape MaxPooling
```
output = input ÷ pool_size   →   24÷2 = 12   (params = 0 siempre)
```

## Flatten
```
output = alto × ancho × canales   →   4×4×64 = 1,024
```

---
## Diagnóstico rápido de gráficas
| Curva | Diagnóstico | Solución |
|-------|------------|---------|
| val_loss ↑, train_loss ↓ | Overfitting | Dropout, EarlyStopping, menos epochs |
| Ambas curvas ↑ desde inicio | Underfitting | Más neuronas, más epochs |
| Oscilaciones grandes | LR muy alto | LR Scheduler, bajar lr |
| Loss = NaN | LR muy alto ó sin normalizar | Normalizar /255, bajar lr |

---
## Activaciones por tipo de salida
| Problema | Activación | Loss |
|---------|-----------|------|
| Regresión | ninguna (lineal) | mean_squared_error |
| Binaria | sigmoid | binary_crossentropy |
| Multiclase | softmax | categorical_crossentropy (one-hot) |
| Multiclase | softmax | sparse_categorical_crossentropy (enteros) |
